In [ ]:
import polars as pl

pl.Config(
    fmt_str_lengths=120,
    fmt_table_cell_list_len=50,
    set_tbl_rows=100,
    tbl_cols=-1,
)

for methode_dividendes in ("flat", "progressif"):
    df: pl.DataFrame = pl.DataFrame()

    for i in range(0, 11):
        benefice_reel = 3000
        ca = 50000
        depenses_pro = 5000
        remuneration_plus_is = ca - depenses_pro - benefice_reel
        is_minimal = benefice_reel * 0.15

        salaires_brut = (remuneration_plus_is - is_minimal) * i / 10

        def taux_dividendes():
            taux = {}
            if methode_dividendes == "flat":
                taux = {"retenue_source": 0.7, "abattement": 0}
            elif methode_dividendes == "progressif":
                taux = {"retenue_source": 0.828, "abattement": 0.6}
            return taux

        def calcul_cout_dividendes():
            return remuneration_plus_is - salaires_brut - is_minimal

        def calcul_dividendes_bruts():
            return calcul_cout_dividendes() / 1.15

        def calcul_cout_remuneration():
            return calcul_dividendes_bruts() * 1.15 + salaires_brut

        def calcul_dividendes_net():
            dividendes_bruts = calcul_dividendes_bruts()
            dividendes_net = dividendes_bruts * taux_dividendes()["retenue_source"]
            return dividendes_net

        def calcul_salaires_net():
            return salaires_brut * 0.563

        def calcul_revenu_imposable():
            return (calcul_salaires_net() * 0.9) + (
                calcul_dividendes_net() * taux_dividendes()["abattement"]
            )

        def calcul_revenu_net():
            return calcul_salaires_net() + calcul_dividendes_net()

        def calcul_benefice():
            return ca - depenses_pro - salaires_brut

        def calcul_is():
            return (calcul_benefice() - is_minimal) * 0.15

        def calcul_ir(revenu):
            tranches_ir = [[11488, 0], [29315, 0.11], [83283, 0.3]]
            revenu_restant = revenu
            ir = 0
            for tranche in tranches_ir:
                if revenu_restant > 0:
                    ir_tranche = min(revenu_restant, tranche[0]) * tranche[1]
                    ir = +ir_tranche
                    revenu_restant = revenu_restant - tranche[0]

            return ir

        def calcul_revenu_ae():
            apres_cotisations = calcul_cout_remuneration() * 0.76 - depenses_pro
            impots = calcul_ir(apres_cotisations * 0.66)
            return apres_cotisations - impots

        def fmt(value):
            return str(int(value))

        # print(salaires_brut)
        # print(calcul_cout_dividendes())
        # print(calcul_dividendes_bruts())
        # print(calcul_is())
        # print("")
        # print("")
        #
        # print(salaires_brut + calcul_dividendes_bruts() + calcul_is(), " == ", remuneration_plus_is)

        # assert salaires_brut + calcul_dividendes_bruts() + calcul_is() == remuneration_plus_is

        row: dict = {
            "Salaires bruts": [fmt(salaires_brut / 12)],
            "Dividendes bruts": [fmt(calcul_cout_dividendes() / 12)],
            "Benefice": [fmt(calcul_benefice())],
            "Benefice réél": [
                fmt(calcul_benefice() - calcul_is() - calcul_dividendes_bruts())
            ],
            "IS": [fmt(calcul_is())],
            "IR": [fmt(calcul_ir(calcul_revenu_imposable()) / 12)],
            "IS + IR": [fmt(calcul_is() + calcul_ir(calcul_revenu_imposable()))],
            "Salaires net": [fmt(calcul_salaires_net() / 12)],
            "Dividendes net": [fmt(calcul_dividendes_net() / 12)],
            "Revenu net ap. impôts": [
                fmt((calcul_revenu_net() - calcul_ir(calcul_revenu_imposable())) / 12)
            ],
            "Revenue AE ap. impôts": [fmt(calcul_revenu_ae() / 12)],
        }

        df = pl.concat([df, pl.from_dict(row)])

    print(df)

In [ ]:
from dash import ALL, Dash, Input, Output, Patch, State, callback, dcc, html

app = Dash()

app.layout = html.Div(
    [
        html.Button("Add Filter", id="add-filter-btn", n_clicks=0),
        html.Div(id="dropdown-container-div", children=[]),
        html.Div(id="dropdown-container-output-div"),
    ]
)


@callback(
    Output("dropdown-container-div", "children"), Input("add-filter-btn", "n_clicks")
)
def display_dropdowns(n_clicks):
    patched_children = Patch()
    new_dropdown = dcc.Dropdown(
        ["NYC", "MTL", "LA", "TOKYO"],
        id={"type": "city-filter-dropdown", "index": n_clicks},
    )
    patched_children.append(new_dropdown)
    return patched_children


@callback(
    Output("dropdown-container-output-div", "children"),
    Input({"type": "city-filter-dropdown", "index": ALL}, "value"),
    State({"type": "city-dynamic-dropdown", "index": ALL}, "id"),
)
def display_output(values, ids):
    return html.Div(
        [html.Div(f"Dropdown {i + 1} = {value}") for (i, value) in enumerate(values)]
        + [html.P(ids)],
    )


if __name__ == "__main__":
    app.run(debug=True)